# 藥物原料、feeder、contextual data對應
1. API：原料為Acetaminophen powder(APAP_P)、feed by PD3
2. MGST：原料為Magnesium stearate(Mag St)、feed by PD7
3. Loctose：原料為Spray dried lactose (L316FF)、feed by PD2
4. Ac-di-sol：Sodium croscarmellose (Ac-di-sol)、feed by PD5
5. Avicel PH102：原料為Microcrystalline cellulose, Avicel PH102，會有兩個批號同時供應、feed by PD1 and PD4

In [2]:
# %load_ext cudf.pandas
import pandas as pd
import numpy as np
import os

In [3]:
# machine data
liw_feeders_1 = pd.read_csv("MSOM data external sharing/Machine data/LiW Feeders 1.csv")
liw_feeders_2 = pd.read_csv("MSOM data external sharing/Machine data/LiW Feeders 2.csv")
blenders = pd.read_csv("MSOM data external sharing/Machine data/Blenders.csv")
pressor = pd.read_csv("MSOM data external sharing/Machine data/Tablet Press.csv")
temperature = pd.read_csv("MSOM data external sharing/Machine data/Temperature.csv")
humidity = pd.read_csv("MSOM data external sharing/Machine data/Humidity.csv")

/tmp/ipykernel_23123/2210807012.py:2: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  liw_feeders_1 = pd.read_csv("MSOM data external sharing/Machine data/LiW Feeders 1.csv")
/tmp/ipykernel_23123/2210807012.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  liw_feeders_2 = pd.read_csv("MSOM data external sharing/Machine data/LiW Feeders 2.csv")
/tmp/ipykernel_23123/2210807012.py:4: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  blenders = pd.read_csv("MSOM data external sharing/Machine data/Blenders.csv")
/tmp/ipykernel_23123/2210807012.py:5: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  pressor =

In [4]:
# concat liw_feeder1&2
liw_feeder = pd.concat([liw_feeders_1, liw_feeders_2], axis=1)
liw_feeder = liw_feeder.loc[:, ~liw_feeder.columns.str.contains("Unnamed:")]

In [5]:
#日期處理
def parse_mixed_date(date_str):
    try:
        # 嘗試美式日期
        return pd.to_datetime(date_str, format="%m/%d/%Y %H:%M")
    except ValueError:
        try:
            # 如果失敗，改用歐洲格式
            return pd.to_datetime(date_str, format="%d/%m/%Y %H:%M")
        except:
            return pd.NaT


liw_feeder["TimeStamp"] = liw_feeder["TimeStamp"].apply(parse_mixed_date)
pressor["TimeStamp"] = pressor["TimeStamp"].apply(parse_mixed_date)


liw_feeder = liw_feeder.dropna(subset=["TimeStamp"], how="any")
pressor = pressor.dropna(subset=['TimeStamp'], how="any")

In [6]:
blenders['TimeStamp'] = blenders['TimeStamp'].map(lambda x: x[:-3] if len(x) > 16 else x)
blenders["TimeStamp"] = blenders["TimeStamp"].apply(parse_mixed_date)
blenders = blenders.dropna(subset=['TimeStamp'], how="any")

In [7]:
# 直接嘗試自動解析，錯誤時返回 NaT
def parse_mixed_date_series(date_series):
    return pd.to_datetime(date_series, errors='coerce', infer_datetime_format=True)

# 批量應用到 DataFrame
for df in [liw_feeder, blenders, pressor]:
    df["TimeStamp"] = parse_mixed_date_series(df["TimeStamp"])
    df.dropna(subset=["TimeStamp"], inplace=True)

/tmp/ipykernel_23123/1360516746.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(date_series, errors='coerce', infer_datetime_format=True)
/tmp/ipykernel_23123/1360516746.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(date_series, errors='coerce', infer_datetime_format=True)
/tmp/ipykernel_23123/1360516746.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consist

In [8]:
# contextual quality
logbook = pd.read_excel("MSOM data external sharing/Contextual quality data/Logbook Long Run Days.xlsx")
content = pd.read_excel("MSOM data external sharing/Contextual quality data/RM Content Uniformity.xlsx", header=1)
material_property = pd.read_excel("MSOM data external sharing/Contextual quality data/RM Material Properties.xlsx")
tablet_property = pd.read_excel("MSOM data external sharing/Contextual quality data/RM Tablet Properties and Drum Change.xlsx", sheet_name="Tablet properties", header=1)
drum_change = pd.read_excel("MSOM data external sharing/Contextual quality data/RM Tablet Properties and Drum Change.xlsx", sheet_name="Raw Material Drum change", header=1)

In [9]:
# tablet_property 處理
tablet_property = tablet_property.dropna(axis=0, how="all")

In [10]:
# drum_change 處理
api = drum_change.iloc[1:, 0:4].dropna(axis=0, how="any")
mgst = drum_change.iloc[1:, 4:8].dropna(axis=0, how="any")
lactose = drum_change.iloc[1:, 8:12].dropna(axis=0, how="any")
Ac_Di_Sol = drum_change.iloc[1:, 12:16].dropna(axis=0, how="any")
Avicel_102_PD1 = drum_change.iloc[1:, 16:20].dropna(axis=0, how="any")
Avicel_102_PD4 = drum_change.iloc[1:, 20:24].dropna(axis=0, how="any")

api = api[api["Est Refill time"] != "missing"]
api = api.reset_index(drop=True)
mgst = mgst.rename(columns={"Date/Time.1": "Date/Time", "Lot.1":"Lot"})
lactose = lactose.rename(columns={"Date/Time.2": "Date/Time"})
Ac_Di_Sol = Ac_Di_Sol.rename(columns={"Date/Time.3": "Date/Time", "Drum #.1": "Drum #"})
Avicel_102_PD1 = Avicel_102_PD1.rename(columns={"Date/Time.4": "Date/Time", "Drum #.2": "Drum #"})
Avicel_102_PD4 = Avicel_102_PD4.rename(columns={"Date/Time.5": "Date/Time", "Drum #.3": "Drum #"})

In [11]:
data_set = {
    "API": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "MGST": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "Lactose": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "Ac_Di_Sol": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "Avicel_102_PD1": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "Avicel_102_PD4": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    }
}

In [12]:
# Based on the sampling interval of output type, find the data points of liw_feeder、blenders、pressor in each time interval.
# 設定時間區間
def time(i: int, input: pd.DataFrame):
    if i == len(input) - 1: 
        return f'After {input.iloc[i]["Date/Time"]}'
    return f'{input.iloc[i+1]["Date/Time"]}'

# 根據時間區間確定那些process data應該要放入
def data_within_time(i: int, input: pd.DataFrame, process: pd.DataFrame):
    # 在input的第一個時間之前的process data
    if i == -1:
        return process[process["TimeStamp"] <= input.iloc[i+1]["Date/Time"]]
    # 在input的最後一個時間之後的process data
    if i == len(input) - 1:
        return process[process["TimeStamp"] >= input.iloc[i-1]["Date/Time"]]

    return process[(process["TimeStamp"] > input.iloc[i]["Date/Time"]) & (process["TimeStamp"] < input.iloc[i+1]["Date/Time"])]

# 標記record的batch
def set_process_data_batch(data: pd.DataFrame, input_key:str, process_key:str, input: pd.DataFrame, process: pd.DataFrame):
    
    batch_name = "Drum #"

    # 如果是API或MGST的話就將batch_name換為Lot
    if input_key == "API" or input_key == "MGST":
        batch_name = "Lot"

    print(input_key, batch_name)
    keys = list(data[input_key][process_key].keys())

    # init batch column
    for key in keys:
        # init batch column
        data[input_key][process_key][key]["Batch"] = 0

        #建立batch的bins
        bins = list(input["Date/Time"].sort_values()) + [pd.Timestamp.max]
        labels = input[batch_name].astype(str)
        
        # input['Date/Time']出現重複的，就drop
        if any(input['Date/Time'].duplicated()):
            bins = list(input["Date/Time"].drop_duplicates().sort_values()) + [pd.Timestamp.max]
            labels = input[batch_name].drop_duplicates().astype(str)


        # 分配對應批次
        data[input_key][process_key][key]["Batch"] = pd.cut(data_set[input_key][process_key][key]["TimeStamp"], bins=bins, labels=labels, right=False, ordered=False)

In [13]:
# 依據input與process將data區分
input_keys = list(data_set.keys())
process_keys = list(data_set['API'].keys())

inputs = [api, mgst, lactose, Ac_Di_Sol, Avicel_102_PD1, Avicel_102_PD4]
process = [liw_feeder, blenders, pressor]

for i, input_key in enumerate(input_keys):
    for j, process_key in enumerate(process_keys):
        k = -1
        while k < len(inputs[i]):
            data_set[input_key][process_key][time(k, inputs[i])] = data_within_time(k, inputs[i], process[j])
            k += 1

for i, input_key in enumerate(input_keys):
    for j, process_key in enumerate(process_keys):
        set_process_data_batch(data_set, input_key, process_key, inputs[i], process[j])

API Lot
API Lot


/tmp/ipykernel_23123/906551961.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = 0
/tmp/ipykernel_23123/906551961.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = pd.cut(data_set[input_key][process_key][key]["TimeStamp"], bins=bins, labels=labels, right=False, ordered=False)
/tmp/ipykernel_23123/906551961.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using

API Lot
MGST Lot


/tmp/ipykernel_23123/906551961.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = pd.cut(data_set[input_key][process_key][key]["TimeStamp"], bins=bins, labels=labels, right=False, ordered=False)
/tmp/ipykernel_23123/906551961.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = 0
/tmp/ipykernel_23123/906551961.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using

MGST Lot
MGST Lot
Lactose Drum #


...

/tmp/ipykernel_23123/906551961.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = pd.cut(data_set[input_key][process_key][key]["TimeStamp"], bins=bins, labels=labels, right=False, ordered=False)
/tmp/ipykernel_23123/906551961.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = 0
/tmp/ipykernel_23123/906551961.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using

Avicel_102_PD1 Drum #
Avicel_102_PD1 Drum #
Avicel_102_PD4 Drum #


/tmp/ipykernel_23123/906551961.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = pd.cut(data_set[input_key][process_key][key]["TimeStamp"], bins=bins, labels=labels, right=False, ordered=False)
/tmp/ipykernel_23123/906551961.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = 0
/tmp/ipykernel_23123/906551961.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using

Avicel_102_PD4 Drum #
Avicel_102_PD4 Drum #


/tmp/ipykernel_23123/906551961.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = pd.cut(data_set[input_key][process_key][key]["TimeStamp"], bins=bins, labels=labels, right=False, ordered=False)
/tmp/ipykernel_23123/906551961.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[input_key][process_key][key]["Batch"] = 0
/tmp/ipykernel_23123/906551961.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using

In [14]:
# 因為最內層的key是由inputs決定，所以process用甚麼都沒關係
api_keys = list(data_set["API"]["liw_feeder"].keys())
mgst_keys = list(data_set["MGST"]["liw_feeder"].keys())
lactose_keys = list(data_set['Lactose']["liw_feeder"].keys())
Ac_Di_Sol_keys = list(data_set['Ac_Di_Sol']["liw_feeder"].keys())
Avicel_102_PD1_keys = list(data_set['Avicel_102_PD1']["liw_feeder"].keys())
Avicel_102_PD4_keys = list(data_set['Avicel_102_PD4']["liw_feeder"].keys())

# 將keys整合成一個二維陣列
output_keys = [api_keys, mgst_keys, lactose_keys, Ac_Di_Sol_keys, Avicel_102_PD1_keys, Avicel_102_PD4_keys]

# Contextual Data的標記
根據contextual sampling的時間做區隔

## Data Concat
1. 將contextual data加入process不同時間段的資料中
2. 將process不同時間段的data frame垂直整合
3. 將input中不同的process橫向整合

In [15]:
# 標記contextual record的batch
def set_contextual_data_bin(input: pd.DataFrame, sampling_bins: list):
    # 在process表格中，加入contextual sampling 的時間
    bins = [pd.Timestamp.min] + list(sampling_bins)
    labels = sampling_bins

    df = input.copy()

    df["sample_time"] = pd.cut(df["TimeStamp"], bins=bins ,labels=labels, right=False, ordered=False)

    return df

In [16]:
# easy for testing
# it should be only have content_first_day = content[:26]

content = pd.read_excel("MSOM data external sharing/Contextual quality data/RM Content Uniformity.xlsx", header=1)
content = content.iloc[:, :-1].drop_duplicates(subset=['Time']).sort_values(by='Time')

In [17]:
result = {
    "API": pd.DataFrame(),
    "MGST": pd.DataFrame(),
    "Lactose": pd.DataFrame(),
    "Ac_Di_Sol": pd.DataFrame(),
    "Avicel_102_PD1": pd.DataFrame(),
    "Avicel_102_PD4": pd.DataFrame()
}

In [18]:
result_keys = list(result.keys())

for i, input_key in enumerate(input_keys):
    for process_key in process_keys:
        process_temp = pd.DataFrame()

        # 設定contextual data的時間區間
        # 並將process中不同的時間區間的dataframe合併
        for output_key in output_keys[i]:
            df = set_contextual_data_bin(data_set[input_key][process_key][output_key], list(content['Time']))
            process_temp = pd.concat([process_temp, df], axis=0)
            
        # 重設index
        result[input_key] = result[input_key].reset_index(drop=True)
        process_temp = process_temp.reset_index(drop=True)

        # 將input的不同process做水平合併
        result[input_key] = pd.concat([result[input_key], process_temp], axis=1)

In [19]:
# 去除重覆的欄位
# 先檢查重覆的欄位所有cell是否相同
def remove_duplicate_columns(df):
    _, unique_idx = np.unique(df.columns, return_index=True)  # 找到唯一欄位的索引
    return df.iloc[:, np.sort(unique_idx)]  # 只保留唯一欄位

# 使用字典推導式來一次性處理所有 DataFrame
result = {key: remove_duplicate_columns(df) for key, df in result.items()}  

# 將result中的na去除
result = {key: df.dropna(axis=0, how="any") for key, df in result.items()}

In [20]:
# 將content uniformity合併進入df
for result_key in result_keys:
    result[result_key]['sample_time'] = pd.to_datetime(result[result_key]['sample_time'].astype(str), errors = 'coerce')
    result[result_key] = result[result_key].merge(content, left_on='sample_time', right_on='Time', how='left')
    result[result_key].drop(columns=['Time'], inplace=True)
    result[result_key].rename(columns={'Sample': 'sample_id'}, inplace=True)

In [21]:
# 將object轉為numeric
for result_key in result_keys:
    for col in result[result_key].select_dtypes(include=['object']).columns:
        result[result_key][col] = pd.to_numeric(result[result_key][col], errors='coerce')

In [22]:
# 依照資料順序將時間調整成依照秒數
for result_key in result_keys:
    result[result_key] = result[result_key].sort_index(ascending=False)

    result[result_key]['Timestamp_Count'] = 59 - result[result_key].groupby('TimeStamp').cumcount()
    result[result_key]['TimeStamp'] = result[result_key]['TimeStamp'] + pd.to_timedelta(result[result_key]['Timestamp_Count'], unit='s')
    result[result_key].drop(columns=['Timestamp_Count'], inplace=True)
    
    # 將df依照TimeStamp排列
    result[result_key] = result[result_key].sort_values(['TimeStamp'], ascending=True)
    
    # 將時間資料轉換成數值欄位
    result[result_key]['second'] = result[result_key]['TimeStamp'].dt.second
    result[result_key]['minute'] = result[result_key]['TimeStamp'].dt.minute
    result[result_key]['hour'] = result[result_key]['TimeStamp'].dt.hour
    result[result_key]['day'] = result[result_key]['TimeStamp'].dt.day
    result[result_key]['month'] = result[result_key]['TimeStamp'].dt.month
    result[result_key]['year'] = result[result_key]['TimeStamp'].dt.year


In [23]:
# 移動重要欄位到最前面
for result_key in result_keys:
    first_cols = ["Batch", "sample_time", "sample_id"]
    cols = first_cols + [col for col in result[result_key] if col not in first_cols]
    result[result_key] = result[result_key][cols]

In [24]:
# 使用中位數填補na_columns
for result_key in result_keys:
    na_columns = result[result_key].columns[result[result_key].isna().any()].to_list()
    result[result_key][na_columns] = result[result_key][na_columns].fillna(result[result_key][na_columns].median())

In [25]:
os.makedirs('processed_data', exist_ok = True)
for result_key in result_keys:
    result[result_key].to_csv(f"processed_data/{result_key}.csv", index=False)